# Gemma 2B Fine-Tuning on Dolly 15k
This notebook fine-tunes `google/gemma-2b-it` on Databricks Dolly 15k and evaluates it with LLM-as-a-judge (GPT-5.2 and DeepSeek).

In [1]:
!pip install -q -U transformers datasets trl accelerate bitsandbytes
!pip install -q tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.8 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Dataset Preparation

- **Training**: 3,000 Dolly samples (no Alpaca in training)
- **ID Test**: 100 held-out Dolly samples
- **OOD Test**: 50 Alpaca samples
- Alpaca columns (`input`, `output`) are normalized to (`context`, `response`) to match Dolly schema.

In [3]:
from datasets import load_dataset

SEED = 15179996

def get_datasets():

    # Dolly 15K dataset
    dolly = load_dataset('databricks/databricks-dolly-15k', split='train')
    dolly = dolly.shuffle(seed=SEED)
    train_dolly = dolly.select(range(3000))          # 3,000 training samples
    test_dolly  = dolly.select(range(3000, 3100))    # 100 held-out ID samples

    # Alpaca (OOD)
    alpaca = load_dataset('tatsu-lab/alpaca', split='train')
    alpaca = alpaca.shuffle(seed=SEED)
    alpaca = alpaca.rename_columns({'input': 'context', 'output': 'response'})
    test_alpaca = alpaca.select(range(50))           # 50 OOD samples

    return train_dolly, test_dolly, test_alpaca


def formatting_prompts_func(examples):
    system_prompt = (
        'You are a helpful, accurate, and concise AI assistant. '
        'Below is an instruction that describes a task, paired with an input that '
        'provides further context. Write a response that appropriately completes the request.'
    )
    prompts = []
    completions = []
    for instruction, context, response in zip(
        examples['instruction'], examples['context'], examples['response']
    ):
        prompt = f'<start_of_turn>user\n{system_prompt}\n\nInstruction: {instruction}'
        if context:
            prompt += f'\n\nContext: {context}'
        prompt += f'<end_of_turn>\n<start_of_turn>model\n'
        prompts.append(prompt)

        completion = f'{response}<end_of_turn>'
        completions.append(completion)

    return {'prompt': prompts, 'completion': completions}


In [4]:
train_dolly, test_dolly, test_alpaca = get_datasets()
train_dataset = train_dolly.map(
    formatting_prompts_func, batched=True, remove_columns=train_dolly.column_names
)
print(f'Train: {len(train_dataset)} | ID Test: {len(test_dolly)} | OOD Test: {len(test_alpaca)}')

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Train: 3000 | ID Test: 100 | OOD Test: 50


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'google/gemma-2b-it'

def load_model_and_tokenizer(model_id=MODEL_ID):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map='auto'
    )
    return model, tokenizer


In [6]:
model, tokenizer = load_model_and_tokenizer()
print(f'Model dtype: {model.dtype} | Params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B')

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model dtype: torch.bfloat16 | Params: 2.51B


In [7]:
import json
import re
import requests
import pandas as pd
from tqdm import tqdm

SYSTEM_PROMPT = (
    'You are a helpful, accurate, and concise AI assistant. '
    'Below is an instruction that describes a task, paired with an input that '
    'provides further context. Write a response that appropriately completes the request.'
)

def generate_responses(model, tokenizer, dataset, output_file):
    """Run greedy inference on dataset; save results to output_file."""
    results = []
    for item in tqdm(dataset):
        prompt = f'<start_of_turn>user\n{SYSTEM_PROMPT}\n\nInstruction: {item["instruction"]}'
        if item.get('context'):
            prompt += f'\n\nContext: {item["context"]}'
        prompt += '<end_of_turn>\n<start_of_turn>model\n'

        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        response = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True
        )
        results.append({
            'instruction': item['instruction'],
            'context': item.get('context', ''),
            'reference_response': item.get('response', ''),
            'model_response': response
        })

    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)
    return results


JUDGE_SYSTEM_PROMPT = '''You are an impartial judge evaluating the quality of an AI response. You will be provided with an instruction, a context (if applicable), and the AI\'s response.
Evaluation Process:
1. Step-by-Step Analysis: Critique the response based on the rubric.
2. Score Assignment: Provide a score for each metric (1-5).
3. Final Verdict: Justify why the scores were given.
Rubric:
Instruction Following: Did it do exactly what was asked? (5 = Perfect, 1 = Irrelevant)
Helpfulness: Is the information useful and correct? (5 = Insightful, 1 = Harmful/Useless)
Fluency: Is the text natural and grammatically sound? (5 = Native-level, 1 = Incoherent)
Please conclude your evaluation with a Markdown table and a brief \'Verdict\' section using the *exact* headers below:
```
## Final Scores
| Metric | Score |
| --- | --- |
| Instruction Following | [1-5] |
| Helpfulness | [1-5] |
| Fluency | [1-5] |
## Verdict
[One sentence summarizing the reasoning for the above scores]
```'''


def call_openrouter(model_name, api_key, instruction, context, response):
    url = 'https://openrouter.ai/api/v1/chat/completions'
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json'
    }
    user_content = f'Instruction: {instruction}\nContext: {context}\nAI Response: {response}'
    payload = {
        'model': model_name,
        'messages': [
            {'role': 'system', 'content': JUDGE_SYSTEM_PROMPT},
            {'role': 'user', 'content': user_content}
        ]
    }
    res = requests.post(url, headers=headers, json=payload, timeout=60)
    res.raise_for_status()
    return res.json()['choices'][0]['message']['content']


def parse_judgement(judgement_text):
    """Robustly parse scores from the judge's markdown table."""
    scores = {}
    metric_keys = {
        'Instruction Following': 'instruction_following',
        'Helpfulness': 'helpfulness',
        'Fluency': 'fluency'
    }
    for label, key in metric_keys.items():
        pattern = rf'\|\s*{re.escape(label)}\s*\|\s*(\d)'
        match = re.search(pattern, judgement_text)
        if match:
            scores[key] = int(match.group(1))
        else:
            scores[key] = None
    return scores


In [ ]:
base_dolly_results  = generate_responses(model, tokenizer, test_dolly,  'base_dolly_evals.json')
base_alpaca_results = generate_responses(model, tokenizer, test_alpaca, 'base_alpaca_evals.json')
print(f'Base Dolly evals: {len(base_dolly_results)} | Base Alpaca evals: {len(base_alpaca_results)}')

100%|██████████| 50/50 [02:38<00:00,  3.16s/it]

Base Dolly evals: 100 | Base Alpaca evals: 50


In [8]:
import torch
from transformers import set_seed
from trl import SFTTrainer, SFTConfig

SEED = 15179996

def set_reproducibility():
    set_seed(SEED)

def get_training_args(output_dir):
    return SFTConfig(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        optim='adamw_torch',
        save_strategy='epoch',
        logging_steps=10,
        bf16=True,
        report_to='none',
        seed=SEED,
        push_to_hub=True,
        hub_model_id='bdanko/fine-tuned-gemma-2b-dolly',

        completion_only_loss=True,
        packing=False
    )

def setup_trainer(model, tokenizer, train_dataset, training_args):
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        args=training_args,
        processing_class=tokenizer,
    )
    return trainer

In [9]:
set_reproducibility()
training_args = get_training_args('gemma-dolly-ft')
trainer = setup_trainer(model, tokenizer, train_dataset, training_args)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Step,Training Loss
10,2.450498
20,1.910918
30,2.173105
40,2.094154
50,1.965380
60,2.017510
70,2.050414
80,1.852354
90,2.007922
100,1.905695


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=564, training_loss=1.1570750580611804, metrics={'train_runtime': 820.7455, 'train_samples_per_second': 10.966, 'train_steps_per_second': 0.687, 'total_flos': 4.451577178093978e+16, 'train_loss': 1.1570750580611804})

In [10]:
# test inference to check for model collapse in responses

import torch
model.eval()
model.config.use_cache = True

for i in range(3):
    sample = test_dolly[i]
    prompt = f'<start_of_turn>user\n{SYSTEM_PROMPT}\n\nInstruction: {sample["instruction"]}'
    if sample.get("context"):
        prompt += f'\n\nContext: {sample["context"]}'
    prompt += "<end_of_turn>\n<start_of_turn>model\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

    print(f"\nSample {i+1}")
    print(f"Instruction: {sample['instruction']}")
    print(f"Response: {response}")


Sample 1
Instruction: Summarize in bullet points some methods used to produce verdigris.
Response: - Hang copper plates over hot vinegar in a sealed pot until a green crust forms.
- Attach copper strips to a wooden block with acetic acid, then bury the block in dung.
- Prepare copper(II) acetate by treatment of copper(II) hydroxide with acetic acid.

Sample 2
Instruction: Extract from the text the year in which Esko Olavi Ahonen was born
Response: Esko Olavi Ahonen was born in 1955 and he is currently 68 years old.

Sample 3
Instruction: In the series A Song of Ice and Fire, who is the founder of House Bar Emmon?
Response: House Bar Emmon is a sept of House Targaryen. The progenitor of the sept was Lysa Bar Emmon, who was the daughter of Barristan and Lysa of the Red Fork.


In [11]:
# save model to huggingface
model.push_to_hub('bdanko/fine-tuned-gemma-2b-dolly')
tokenizer.push_to_hub('bdanko/fine-tuned-gemma-2b-dolly')

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...v_0ww_k/model.safetensors:   3%|3         |  152MB / 5.01GB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpam22duz6/tokenizer.json: 100%|##########| 34.3MB / 34.3MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/bdanko/fine-tuned-gemma-2b-dolly/commit/64a3a7c58b6dc85ba6f2be0d14cd401d1715893e', commit_message='Upload tokenizer', commit_description='', oid='64a3a7c58b6dc85ba6f2be0d14cd401d1715893e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/bdanko/fine-tuned-gemma-2b-dolly', endpoint='https://huggingface.co', repo_type='model', repo_id='bdanko/fine-tuned-gemma-2b-dolly'), pr_revision=None, pr_num=None)

In [12]:
# Evaluating the fine tuned models

ft_dolly_results  = generate_responses(model, tokenizer, test_dolly,  'ft_dolly_evals.json')
ft_alpaca_results = generate_responses(model, tokenizer, test_alpaca, 'ft_alpaca_evals.json')
print(f'FT Dolly evals: {len(ft_dolly_results)} | FT Alpaca evals: {len(ft_alpaca_results)}')

100%|██████████| 50/50 [01:15<00:00,  1.52s/it]

FT Dolly evals: 100 | FT Alpaca evals: 50


In [14]:
import json
from huggingface_hub import HfApi

eval_files = [
    'base_dolly_evals.json',
    'base_alpaca_evals.json',
    'ft_dolly_evals.json',
    'ft_alpaca_evals.json'
]
repo_id = "bdanko/fine-tuned-gemma-2b-dolly"
api = HfApi()

for filename in eval_files:
    try:
        with open(filename, 'r') as f:
            var_name = filename.replace('_evals.json', '_results')
            globals()[var_name] = json.load(f)

            api.upload_file(
                path_or_fileobj=filename,
                path_in_repo=f"eval_results/{filename}",
                repo_id=repo_id,
                repo_type="model"
            )
            print(f"Uploaded {filename} to {repo_id}/eval_results/")

    except FileNotFoundError:
        print(f"{filename} not found locally, skipping.")
    except Exception as e:
        print(f"Error processing {filename}: {e}")


Uploaded base_dolly_evals.json to bdanko/fine-tuned-gemma-2b-dolly/eval_results/
Uploaded base_alpaca_evals.json to bdanko/fine-tuned-gemma-2b-dolly/eval_results/
Uploaded ft_dolly_evals.json to bdanko/fine-tuned-gemma-2b-dolly/eval_results/
Uploaded ft_alpaca_evals.json to bdanko/fine-tuned-gemma-2b-dolly/eval_results/


## 7. LLM Judging

Both judges evaluate **the same outputs**. Each result entry stores raw text and structured scores.

In [ ]:
api_key = os.environ['OPENROUTER_API_KEY']
JUDGES = ['openai/gpt-5.2', 'deepseek/deepseek-v3.2']

all_evals = {
    'base_dolly':  base_dolly_results,
    'base_alpaca': base_alpaca_results,
    'ft_dolly':    ft_dolly_results,
    'ft_alpaca':   ft_alpaca_results,
}

judgement_results = {}
MAX_RETRIES = 3

for key, results in all_evals.items():
    judgement_results[key] = []
    for res in tqdm(results, desc=f'Judging {key}'):
        item_judgements = {}
        for judge in JUDGES:
            raw = ''
            scores = {'instruction_following': None, 'helpfulness': None, 'fluency': None}
            
            for attempt in range(MAX_RETRIES):
                try:
                    raw = call_openrouter(
                        judge, api_key,
                        res['instruction'], res['context'], res['model_response']
                    )
                    scores = parse_judgement(raw)
                    
                    # Check if all scores were successfully parsed
                    if all(v is not None for v in scores.values()):
                        break
                    else:
                        print(f'\n[Attempt {attempt+1}] Parsing failed for {judge}. Retrying...')
                except Exception as e:
                    print(f'\n[Attempt {attempt+1}] API Error for {judge}: {e}')
                    import time
                    time.sleep(2) # Backoff
            
            item_judgements[judge] = {'raw': raw, 'scores': scores}
            
        judgement_results[key].append({
            'instruction':    res['instruction'],
            'context':        res['context'],
            'model_response': res['model_response'],
            'judgements':     item_judgements
        })

with open('judgement_results.json', 'w') as f:
    json.dump(judgement_results, f, indent=2)
print('Judging complete.')

In [ ]:
summary_data = []
for key, entries in judgement_results.items():
    model_label = 'base' if 'base' in key else 'fine-tuned'
    dist_label  = 'in'   if 'dolly' in key else 'out'
    for judge in JUDGES:
        valid = [e for e in entries if e['judgements'][judge]['scores']['instruction_following'] is not None]
        n = len(valid) or 1
        avg_if = sum(e['judgements'][judge]['scores']['instruction_following'] for e in valid) / n
        avg_h  = sum(e['judgements'][judge]['scores']['helpfulness']           for e in valid) / n
        avg_f  = sum(e['judgements'][judge]['scores']['fluency']               for e in valid) / n
        summary_data.append({
            'Model':                 model_label,
            'Distribution':          dist_label,
            'Judge':                 judge,
            'Instruction Following': round(avg_if, 2),
            'Helpfulness':           round(avg_h,  2),
            'Fluency':               round(avg_f,  2),
        })

df = pd.DataFrame(summary_data)
df = df.sort_values(['Model', 'Distribution', 'Judge']).reset_index(drop=True)
print(df.to_markdown(index=False))

In [ ]:
from datasets import Dataset

def upload_judgements(judgement_list, judge_name, repo_id):
    """Upload per-judge judgement results as a HF Dataset."""
    rows = []
    for entry in judgement_list:
        j = entry['judgements'].get(judge_name, {})
        rows.append({
            'instruction':            entry['instruction'],
            'context':                entry['context'],
            'model_response':         entry['model_response'],
            'raw_judgement':          j.get('raw', ''),
            'instruction_following':  j.get('scores', {}).get('instruction_following'),
            'helpfulness':            j.get('scores', {}).get('helpfulness'),
            'fluency':                j.get('scores', {}).get('fluency'),
        })
    ds = Dataset.from_list(rows)
    ds.push_to_hub(repo_id)
    print(f'Uploaded {len(ds)} rows → {repo_id}')

# judge datasets
upload_judgements(judgement_results['base_dolly'],  'openai/gpt-5.2',
                  'bdanko/base-gemma-2b-dolly-evaluation-gpt-5.2-judgement')
upload_judgements(judgement_results['base_dolly'],  'deepseek/deepseek-v3.2',
                  'bdanko/base-gemma-2b-dolly-evaluation-deepseek-v3.2-judgement')
upload_judgements(judgement_results['base_alpaca'], 'openai/gpt-5.2',
                  'bdanko/base-gemma-2b-alpaca-evaluation-gpt-5.2-judgement')
upload_judgements(judgement_results['base_alpaca'], 'deepseek/deepseek-v3.2',
                  'bdanko/base-gemma-2b-alpaca-evaluation-deepseek-v3.2-judgement')
upload_judgements(judgement_results['ft_dolly'],    'openai/gpt-5.2',
                  'bdanko/fine-tuned-gemma-2b-dolly-evaluation-gpt-5.2-judgement')
upload_judgements(judgement_results['ft_dolly'],    'deepseek/deepseek-v3.2',
                  'bdanko/fine-tuned-gemma-2b-dolly-evaluation-deepseek-v3.2-judgement')
upload_judgements(judgement_results['ft_alpaca'],   'openai/gpt-5.2',
                  'bdanko/fine-tuned-gemma-2b-alpaca-evaluation-gpt-5.2-judgement')
upload_judgements(judgement_results['ft_alpaca'],   'deepseek/deepseek-v3.2',
                  'bdanko/fine-tuned-gemma-2b-alpaca-evaluation-deepseek-v3.2-judgement')

print('All deliverables uploaded to HuggingFace Hub.')

## 10. Qualitative Assessments

Find examples programmatically, then annotate below.

In [ ]:
# Compare avg score across both judges: base vs fine-tuned

def avg_score(entry, judge):
    s = entry['judgements'][judge]['scores']
    vals = [v for v in s.values() if v is not None]
    return sum(vals) / len(vals) if vals else 0


def total_avg(entry):
    return sum(avg_score(entry, j) for j in JUDGES) / len(JUDGES)


base_lookup = {e['instruction']: e for e in judgement_results['base_dolly']}
ft_lookup   = {e['instruction']: e for e in judgement_results['ft_dolly']}

deltas = []
for instr, ft_entry in ft_lookup.items():
    if instr in base_lookup:
        base_score = total_avg(base_lookup[instr])
        ft_score   = total_avg(ft_entry)
        deltas.append((ft_score - base_score, instr, base_lookup[instr], ft_entry))

deltas.sort(key=lambda x: x[0], reverse=True)

print('=== TOP 3 IMPROVEMENT CASES ===')
for delta, instr, base_e, ft_e in deltas[:3]:
    print(f'\nDelta: +{delta:.2f}')
    print(f'Instruction: {instr[:120]}')
    print(f'Base response: {base_e["model_response"][:300]}')
    print(f'FT response:   {ft_e["model_response"][:300]}')
    print('---')

print('\n=== FAILURE CASE (largest regression) ===')
delta, instr, base_e, ft_e = deltas[-1]
print(f'Delta: {delta:.2f}')
print(f'Instruction: {instr[:120]}')
print(f'Base response: {base_e["model_response"][:300]}')
print(f'FT response:   {ft_e["model_response"][:300]}')

### Improvement Cases

> Fill in after running the cell above. Copy the top 3 improvement examples here with explanation.

1. **Case 1**: [Instruction] — Base model [description of failure]. Fine-tuned model [description of improvement]. Root cause: domain alignment.
2. **Case 2**: [Instruction] — ...
3. **Case 3**: [Instruction] — ...

### Failure Case

> Fill in after running the cell above. Copy the bottom regression example here.

1. **Failure**: [Instruction] — Fine-tuned model scored lower than base. Root cause: [e.g. catastrophic forgetting on long-form reasoning / over-fitting to Dolly response style / ...]